In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ===============================
# 1️⃣ Initialiser Spark
# ===============================


spark = SparkSession.builder \
    .appName("Gestion_logistique") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .getOrCreate()

# ===============================
# 2️⃣ Charger les données
# ===============================
path_data = "./data/processed/DataCoSupplyChain_final.csv"
df = spark.read.csv(path_data, header=True, inferSchema=True)
df.groupBy("Late_delivery_risk").count().show()
# # df.show()
# numeric_cols = ["Order Item Quantity", "Sales", "Order Profit Per Order",
#                 "Product Price", "distance_km", "order_month"]
# categorical_cols = ["Type", "Category Name", "Customer Segment",
#                     "Order Region", "Shipping Mode"]
# target_col = "Late_delivery_risk"

# # ===============================
# # 3️⃣ Gestion des valeurs manquantes
# # ===============================


# # ===============================
# # 4️⃣ Encodage et assemblage
# # ===============================
# indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx") for c in categorical_cols]
# encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec") for c in categorical_cols]

# assembler = VectorAssembler(
#     inputCols=[c+"_vec" for c in categorical_cols] + numeric_cols,
#     outputCol="features"
# )

# # ===============================
# # 5️⃣ Random Forest Classifier
# # ===============================
# rf = RandomForestClassifier(
#     featuresCol="features",
#     labelCol=target_col,
#     seed=42
# )

# # ===============================
# # 6️⃣ Pipeline
# # ===============================
# pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

# # ===============================
# # 7️⃣ Train/Test split
# # ===============================
# train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# # ===============================
# # 8️⃣ GridSearch avec CrossValidator
# # ===============================
# paramGrid = ParamGridBuilder() \
#     .addGrid(rf.numTrees, [50, 100, 200]) \
#     .addGrid(rf.maxDepth, [5, 10, 15]) \
#     .addGrid(rf.maxBins, [32, 64]) \
#     .build()

# evaluator = MulticlassClassificationEvaluator(
#     labelCol=target_col,
#     predictionCol="prediction",
#     metricName="accuracy"
# )

# cv = CrossValidator(
#     estimator=pipeline,
#     estimatorParamMaps=paramGrid,
#     evaluator=evaluator,
#     numFolds=3
# )

# # ===============================
# # 9️⃣ Entraîner le modèle avec GridSearch
# # ===============================
# cv_model = cv.fit(train_df)

# # ===============================
# # 🔟 Prédictions
# # ===============================
# predictions = cv_model.transform(test_df)
# predictions.select("Late_delivery_risk", "prediction", "features").show(10, truncate=False)

# # ===============================
# # 1️⃣1️⃣ Évaluation
# # ===============================
# accuracy = evaluator.evaluate(predictions)
# print("Accuracy:", accuracy)

# f1_evaluator = MulticlassClassificationEvaluator(
#     labelCol=target_col,
#     predictionCol="prediction",
#     metricName="f1"
# )
# f1_score = f1_evaluator.evaluate(predictions)
# print("F1 Score:", f1_score)

# # ===============================
# # 1️⃣2️⃣ Sauvegarde du modèle
# # ===============================
# cv_model.write().overwrite().save("./rf_pipeline_late_delivery")


+------------------+------+
|Late_delivery_risk| count|
+------------------+------+
|                 1|102717|
|                 0| 76163|
+------------------+------+



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler,StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ===============================
# 1️⃣ Initialiser Spark
# ===============================


spark = SparkSession.builder \
    .appName("Gestion_logistique") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .getOrCreate()

# ===============================
#  Charger les données
# ===============================
path_data = "./data/processed/DataCoSupplyChain_final.csv"
df = spark.read.csv(path_data, header=True, inferSchema=True)
df.groupBy("Late_delivery_risk").count().show()
# # df.show()
numeric_cols = ["Order Item Quantity", "Sales", "Order Profit Per Order",
                "Product Price", "distance_km", "order_month"]
categorical_cols = ["Type", "Category Name", "Customer Segment",
                    "Order Region", "Shipping Mode"]
target_col = "Late_delivery_risk"


indexers = [StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCols=[col + "_idx"], outputCols=[col + "_ohe"]) for col in categorical_cols]

#  Assemblage des features
assembler_inputs = [col + "_ohe" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

#  Normalisation (optionnelle)
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

#  Définition du modèle de classification
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="Late_delivery_risk",
    numTrees=100,
    maxDepth=10,
    seed=42
)

# Construction du pipeline MLlib
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

#  Séparation du dataset
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

#  Entraînement du modèle
model = pipeline.fit(train_df)

#  Prédictions
predictions = model.transform(test_df)

#  Évaluation

evaluator = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="Late_delivery_risk",
    predictionCol="prediction",
    metricName="f1"
)

f1_score = f1_evaluator.evaluate(predictions)
print(f"✅ F1-score du modèle : {f1_score:.4f}")

#  Sauvegarde du modèle
model.write().overwrite().save("./models/logistics_rf_pipeline")



/usr/local/lib/python3.10/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/14 00:14:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/14 00:14:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+------------------+------+
|Late_delivery_risk| count|
+------------------+------+
|                 1|102717|
|                 0| 76163|
+------------------+------+



25/11/14 00:15:19 WARN DAGScheduler: Broadcasting large task binary with size 1349.5 KiB
25/11/14 00:15:20 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
25/11/14 00:15:21 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/11/14 00:15:22 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
25/11/14 00:15:25 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
25/11/14 00:15:26 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB


Accuracy: 0.6973669467787115
✅ F1-score du modèle : 0.6962


25/11/14 09:35:22 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 333759 ms exceeds timeout 120000 ms
25/11/14 09:35:22 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/14 09:52:06 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$